# Plain Language Government Navigator - Gemma 4 E4B Fine-tuning

Fine-tunes Gemma 4 E4B-it using Unsloth + QLoRA on a single P100 GPU (16 GB).

**Strategy:** Ultra-tight memory budget — short sequences, minimal LoRA rank, only 2 target modules.

## 1. Setup

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 2. Imports

In [ ]:
import os, gc, json, glob
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 3. Load Model

Minimal memory footprint: seq_len=256, LoRA on only q_proj + v_proj (not all 7 modules).

In [ ]:
MAX_SEQ_LENGTH = 256  # Minimal to save KV cache memory

torch.cuda.empty_cache()
gc.collect()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.float16,
)

print(f"After load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB / {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Minimal LoRA — only attention Q and V projections
model = FastLanguageModel.get_peft_model(
    model,
    r=4,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

model.print_trainable_parameters()
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3
print(f"Free VRAM for training: {free:.2f} GB")

## 4. Load Training Data

Training data is a JSONL file where each line contains a `messages` array in OpenAI chat format:

```json
{"messages": [
    {"role": "system", "content": "You are a plain-language benefits navigator..."},
    {"role": "user", "content": "I lost my job and have two kids..."},
    {"role": "assistant", "content": "Here are programs you may be eligible for..."}
]}
```

Upload the dataset as a Kaggle dataset and reference it via the standard input path.

In [ ]:
import glob

# Find the training data file — Kaggle may nest under dataset slug
candidates = glob.glob("/kaggle/input/**/final.jsonl", recursive=True)
if candidates:
    DATASET_PATH = candidates[0]
else:
    # Fallback: try the direct path
    DATASET_PATH = "/kaggle/input/mn-navigator-training/final.jsonl"

print(f"Dataset path: {DATASET_PATH}")

# Parse JSONL into list of dicts
records = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

dataset = Dataset.from_list(records)

print(f"Training examples: {len(dataset)}")
print(f"\nFirst example roles: {[m['role'] for m in dataset[0]['messages']]}")

## 5. Training

Batch size 1, gradient accumulation 4, fp16. Minimal memory footprint.

In [ ]:
OUTPUT_DIR = "/kaggle/working/navigator-finetune"

torch.cuda.empty_cache()
gc.collect()

# Pre-format dataset
formatted_texts = []
for ex in records:
    text = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )
    formatted_texts.append(text)

formatted_dataset = Dataset.from_dict({"text": formatted_texts})
print(f"Formatted {len(formatted_dataset)} examples")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=50,
    save_total_limit=1,
    fp16=True,
    bf16=False,
    optim="adamw_8bit",
    seed=42,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
)

free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3
print(f"Free VRAM before training: {free:.2f} GB")

## 6. Train

In [ ]:
train_result = trainer.train()
metrics = train_result.metrics
print(f"\nTraining complete!")
print(f"  Training loss: {metrics.get('train_loss', 'N/A'):.4f}")
print(f"  Runtime: {metrics.get('train_runtime', 0):.1f}s")

## 7. Save & Export

In [ ]:
FINAL_DIR = "/kaggle/working/navigator-finetune/final"
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"LoRA adapters saved to: {FINAL_DIR}")
for f in sorted(os.listdir(FINAL_DIR)):
    size_mb = os.path.getsize(os.path.join(FINAL_DIR, f)) / 1024 / 1024
    print(f"  {f} ({size_mb:.1f} MB)")

## 8. Export to GGUF

In [ ]:
GGUF_DIR = "/kaggle/working/navigator-gguf"

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"GGUF saved to: {GGUF_DIR}")
for f in sorted(os.listdir(GGUF_DIR)):
    size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1024 / 1024
    print(f"  {f} ({size_mb:.1f} MB)")

## 9. Modelfile for Ollama

In [ ]:
gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
gguf_filename = gguf_files[0] if gguf_files else "model.gguf"

SYSTEM_PROMPT = (
    "You are a plain-language government benefits navigator for Minnesota. "
    "You help people understand which government assistance programs they may "
    "be eligible for based on their situation. Be warm, clear, and actionable. "
    "Always cite specific eligibility thresholds and application portals. "
    'Never say someone "qualifies" — say "may be eligible." '
    "End every response with a disclaimer that this is informational, not legal advice."
)

modelfile = f"""FROM ./{gguf_filename}
PARAMETER temperature 1.0
PARAMETER top_p 0.95
PARAMETER num_ctx 1024
SYSTEM "{SYSTEM_PROMPT}"
"""

with open(os.path.join(GGUF_DIR, "Modelfile"), "w") as f:
    f.write(modelfile)

print("Modelfile written.")
print("To create Ollama model:")
print(f"  ollama create navigator-e4b -f {GGUF_DIR}/Modelfile")

## 10. Test Inference

In [ ]:
FastLanguageModel.for_inference(model)

test_messages = [{"role": "user", "content":
    "I'm a single mom with two kids ages 3 and 7. I just lost my job "
    "last week and I live in Hennepin County. What help is available?"}]

inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=1.0, top_p=0.95, do_sample=True)

print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))